# Spatial privacy: measure a hypothetical perturbation

What does moving a point protect, and what analytical relationships does it disturb? We apply an explicitly hypothetical, fixed-seed perturbation to historical locations. Displacement is observable; a privacy guarantee is not.

## Objectives

- Construct a reproducible area-uniform annular perturbation
- Compare displacement and aggregate utility
- Explain why masking and aggregation require threat-model evaluation

## Start in a fresh kernel

Run every code cell from the top. The Book shows publication-time outputs;
JupyterLite uses the Python (Pyodide) kernel and downloads its runtime and
packages initially. All observations and metadata below come from packaged
snapshots: no live API, token, paid service or GPU is required. If a file is
missing or modified, validation fails explicitly; restore the supplied files.
Download notebook edits and exports to keep them outside browser storage.
The Python code is visible so that transformations can be inspected.

In [ ]:
import sys
from pathlib import Path

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])

# Book starts here; desktop/Lite may start at either the notebook or content root.
candidates = [Path.cwd(), Path.cwd() / "repository_examples",
              Path.cwd() / "content" / "repository_examples"]
lesson_dir = next((p for p in candidates if (p / "repo_examples.py").is_file()), None)
if lesson_dir is None:
    raise FileNotFoundError("Open this lesson from content/repository_examples on desktop, or repository_examples in Lite, with repo_examples.py and data/ alongside it.")
sys.path.insert(0, str(lesson_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import repo_examples as rx

if sys.platform != "emscripten":
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
display(rx.validate_all())

## Evidence and attribution

The cards describe the evidence type before any visualization. The manifest retains exact URLs/queries, retrieval UTC, SHA-256 checksums, field order, row counts and transformation recipes. Dataset terms remain separate from the undecided project license.

In [ ]:
display(rx.source_card("cholera"))

In [ ]:
deaths = rx.read_csv("cholera/cholera_deaths.csv")
pumps = rx.read_csv("cholera/pumps.csv")
R = 6_371_000
lon0, lat0 = deaths.LON.mean(), deaths.LAT.mean()
original = np.column_stack([np.radians(deaths.LON-lon0)*R*np.cos(np.radians(lat0)), np.radians(deaths.LAT-lat0)*R])
pump_xy = np.column_stack([np.radians(pumps.LON-lon0)*R*np.cos(np.radians(lat0)), np.radians(pumps.LAT-lat0)*R])
weights = deaths.DEATHS.to_numpy()
rng = np.random.default_rng(1854)
r_min, r_max = 50, 150  # meters; hypothetical teaching choices
angle = rng.uniform(0, 2*np.pi, len(deaths))
# sqrt produces uniform area within the annulus; uniform radius does not.
radius = np.sqrt(rng.uniform(r_min**2, r_max**2, len(deaths)))
masked = original + np.column_stack([radius*np.cos(angle), radius*np.sin(angle)])
displacement = np.linalg.norm(masked-original, axis=1)
assert np.all((displacement >= r_min) & (displacement <= r_max))
display(pd.DataFrame({"evidence": ["Historical input", "Hypothetical change", "Derived measurement"],
                      "unit": ["250 locations; 489 recorded deaths", "50–150 m annulus, fixed seed 1854", "Euclidean displacement in approximate local m"]}))

## View 1: show the transformation explicitly

The map arrows link original and perturbed locations in this teaching notebook. Publishing such links for sensitive current records would disclose the transformation; this is a historical demonstration, not a release recipe. Locations can move across streets or other meaningful boundaries. The local planar approximation is the same small-area equirectangular construction used in the cholera lesson.

**Method connection:** the catalog's Scatterplot (§9) and Histogram (§7) help inspect locations and displacement. Paired maps, movement arrows and fixed-grid differences are additional constructions that make the perturbation and its consequences visible. Privacy requires a threat model, external linkage analysis, governance and utility evaluation beyond a visually blurred map.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, points, title in zip(axes, [original, masked], ["Historical locations", "Hypothetical perturbed locations"]):
    ax.scatter(points[:, 0], points[:, 1], s=weights*8, alpha=.5, color="#226e8b")
    ax.scatter(pump_xy[:, 0], pump_xy[:, 1], marker="^", color="#aa4c35", s=30)
    ax.set_aspect("equal")
    rx.finish(ax, title, "Approximate east (m)", "Approximate north (m)")
axes[1].quiver(original[::10, 0], original[::10, 1], (masked-original)[::10, 0], (masked-original)[::10, 1],
               angles="xy", scale_units="xy", scale=1, width=.003, color="black")
fig.tight_layout()
plt.show()

## View 2: displacement is a measurement, not anonymity

The histogram checks the geometric construction. An area-uniform annulus has more area at larger radii, so the displacement histogram need not be flat. The cumulative curve answers the fraction displaced by at most a chosen threshold. Neither indicates a re-identification probability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(displacement, bins=np.linspace(50, 150, 11), color="#3b8199", edgecolor="white")
rx.finish(axes[0], "Hypothetical displacement", "Distance (m)", "Mapped locations")
axes[1].step(np.sort(displacement), np.arange(1, len(deaths)+1)/len(deaths), where="post")
rx.finish(axes[1], "Empirical displacement distribution", "Distance (m)", "Cumulative fraction of locations")
fig.tight_layout()
plt.show()
print(pd.Series(displacement).describe().round(2))

## View 3: aggregation utility depends on scale and origin

Both point sets have the same total weight. A fixed square grid makes their differences comparable; separate adaptive hexbin extents would confound this comparison. The L1 difference sums absolute cell-weight changes and can count a moved death twice, once leaving and once entering a cell. The metric is therefore normalized by twice the unchanged total. Grid cells are illustrative geometry, not administrative wards.

In [ ]:
lower = np.floor(np.minimum(original.min(axis=0), masked.min(axis=0))/100)*100 - 100
upper = np.ceil(np.maximum(original.max(axis=0), masked.max(axis=0))/100)*100 + 100
results = []
for cell_m in [100, 200, 400]:
    edges = [np.arange(lower[d], upper[d]+cell_m, cell_m) for d in [0, 1]]
    a, _, _ = np.histogram2d(*original.T, bins=edges, weights=weights)
    b, _, _ = np.histogram2d(*masked.T, bins=edges, weights=weights)
    assert a.sum() == b.sum() == 489
    results.append({"cell_width_m": cell_m, "normalized_L1_difference": np.abs(a-b).sum()/(2*weights.sum())})
    if cell_m == 200:
        before, after, chosen_edges = a, b, edges
display(pd.DataFrame(results))
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
extent = [chosen_edges[0][0], chosen_edges[0][-1], chosen_edges[1][0], chosen_edges[1][-1]]
maximum = max(before.max(), after.max())
for ax, grid, title in zip(axes[:2], [before, after], ["Original historical weights", "Hypothetical masked weights"]):
    im = ax.imshow(grid.T, origin="lower", extent=extent, cmap="YlOrRd", vmin=0, vmax=maximum)
    rx.finish(ax, title, "East (m)", "North (m)")
diff = after-before
delta = axes[2].imshow(diff.T, origin="lower", extent=extent, cmap="RdBu_r", vmin=-abs(diff).max(), vmax=abs(diff).max())
rx.finish(axes[2], "Masked − original", "East (m)", "North (m)")
fig.colorbar(im, ax=list(axes[:2]), label="Recorded deaths per 200 m cell", shrink=.65)
fig.colorbar(delta, ax=axes[2], label="Change in recorded weight", shrink=.65)
plt.show()
old_nearest = np.hypot(original[:, None, 0]-pump_xy[None, :, 0], original[:, None, 1]-pump_xy[None, :, 1]).argmin(axis=1)
new_nearest = np.hypot(masked[:, None, 0]-pump_xy[None, :, 0], masked[:, None, 1]-pump_xy[None, :, 1]).argmin(axis=1)
print(f"Nearest-pump geometric assignment changed for {(old_nearest != new_nearest).mean():.1%} of locations.")

## Exercise

Repeat with seeds 7 and 21, then shift the grid origin by 100 m. Compare the normalized L1 differences and nearest-pump changes. State the analytical use you are trying to preserve and propose a threat model; identify what this notebook does not measure before recommending any release.

## Limitations and reflection

The input is a historical digitization, not current identifiable health records. Perturbed locations and grid cells are hypothetical, not observed geography. Fixed seeds and bounded displacement do not guarantee anonymity, differential privacy or resistance to linkage. Repeated releases, auxiliary data and rare locations can defeat masking. Aggregation can also disclose small counts. There are no modern ward boundaries, building footprints or clinical records here.

## Citations and repository connections

[Historical GPL data provenance](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/tree/dda3a4667a1267b5ac4858723c46d8c0c82cb7a6/book/notebooks/data/cholera). Teaching question inspired by [PHI ethical geospatial practice at 9cd3b73ca2a3](https://github.com/PHI-Case-Studies/2023-Ethical-Practice-Geospatial-Data-Science/tree/9cd3b73ca2a3630335cd74753e38b819871e3974). Its absent general license is recorded in the atlas; no authored source or unsourced modern geometry was copied.

See the [companion index](index.md) and [full attribution](https://jltobias.github.io/JupyterLite-Data-Visualization/book/references.html). These are original teaching code and narrative; linked repository notebooks and authored figures have not been copied.